- Remove or group features with large cardinality of categories
- encode with one-hot encoding

In [1]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()
df: pd.DataFrame = read_csv(
    filepath=config.data_dir / 'datasets' / 'median' / 'median_imputed.csv',
    dtypes_filepath=config.data_dir / 'datasets' / 'median' / 'median_imputed_dtypes.csv',
    index_list=[0]
)

In [2]:
remove_columns = [
    'TR.HQMinorRegion', # TR.HQCountryCode is more precise
    'TR.HeadquartersRegionAlt', # TR.HQCountryCode is more precise
    'TR.HeadquartersCity', # high cardinality
    'Currency Code', # TR.HQCountryCode is more precise
    'TR.ESGPeriodLastUpdateDate', # not useful
    'TR.RelatedOrgISO2', # related to TR.HQCountryCode
    'TR.BusinessSector', # high cardinality. Focus on GICS codes
    'TR.PriceMainIndex', # Stock Index seems irrelevant
    'TR.InstrumentType', # equal to TR.AssetCategory
    'TR.ISO14000', # Claiming to have an environment certificate of ISO 14000 or EMS seems not relev
    'TR.BusinessSectorScheme', # focus on GICS codes
    'TR.PrivateEquityBacked', # same as TR.PEBackedStatus
]

# Optional
score_columns = [ # Grades from A+ to D-. Could be irrelevant or could be reduced to A-D
    'TR.TRESGHumanRightsScoreGrade',
    'TR.TRESGWorkforceScoreGrade',
    'TR.TRESGShareholdersScoreGrade',
    'TR.TRESGProductResponsibilityScoreGrade',
    'TR.TRESGManagementScoreGrade',
    'TR.TRESGInnovationScoreGrade',
    'TR.TRESGResourceUseScoreGrade',
    'TR.TRESGEmissionsScoreGrade',
    'TR.TRESGCScoreGrade',
    'TR.SocialPillarScoreGrade',
    'TR.EnvironmentPillarScoreGrade',
    'TR.GovernancePillarScoreGrade',
    'TR.TRESGCommunityScoreGrade'
]

# Optional
maybe_irrelevant_columns = [
    'TR.CO2EstimationMethod', # 98% of values are 'Reported' out of 4
    'TR.Scope1EstMethod', # 91% of values are 'Reported' out of 5
    'TR.Scope2EstMethod', # 88% of values are 'Reported' out of 5
    'TR.Scope3EstUpstreamMethod', # 91% of values are 'Reported' out of 6
    'TR.Scope3EstDownstreamMethod', # 54% of values are 'reported_value' out of 10
    'TR.OrganizationSubtype', # 83% of values are 'Operating Company' out of 8
    'TR.AssetCategory', # 92% of values are 'Ordinary Share' out of 16
    'TR.CompanyParentType', # 83% are 'Ultimate Parent' out of 4
    'TR.PEBackedStatus', # 78% of values are 'Never' 19% 'Formerly' out of 4
    'TR.BoardStructureType', # 74% of values are 'Unitary', 14% 'Mixed' and 11% 'Two-tier' out of 3
    'TR.OrganizationType', # 96% of values are 'Business Organization' out of 3
    'TR.RelatedOrgType', # 74% of values are 'Affiliate' and 24% 'Subsidiary' out of 3
]

df.drop(remove_columns, axis=1, inplace=True)

In [3]:
cat_cols = df.select_dtypes(include=['category']).columns

cardinality = df[cat_cols].nunique().sort_values(ascending=False)

n_rows = len(df)
high_card_cols = cardinality[
    (cardinality > 50) | (cardinality > round(0.01 * n_rows))
]

print("High-cardinality columns:")
print(cardinality)

High-cardinality columns:
Instrument                                 2835
TR.HQCountryCode                             66
TR.AssetCategory                             16
TR.TRESGEmissionsScoreGrade                  12
TR.TRESGWorkforceScoreGrade                  12
TR.TRESGShareholdersScoreGrade               12
TR.TRESGResourceUseScoreGrade                12
TR.TRESGProductResponsibilityScoreGrade      12
TR.TRESGInnovationScoreGrade                 12
TR.TRESGHumanRightsScoreGrade                12
TR.TRESGManagementScoreGrade                 12
TR.TRESGCommunityScoreGrade                  12
TR.TRESGCScoreGrade                          12
TR.SocialPillarScoreGrade                    12
TR.GovernancePillarScoreGrade                12
TR.EnvironmentPillarScoreGrade               12
TR.Scope3EstDownstreamMethod                 10
TR.OrganizationSubtype                        8
TR.Scope3EstUpstreamMethod                    6
TR.Scope2EstMethod                            5
TR.Scope1EstMe

In [4]:
import pandas as pd

def top_value_share(dataframe: pd.DataFrame, threshold: float = 0.0) -> pd.DataFrame:
    """
    Return, for each column, the relative frequency (in %) of its most
    frequent value. Only keep columns where this percentage >= threshold.
    """
    # compute top frequency in percent for each column
    top_pct = dataframe.apply(
        lambda s: s.value_counts(normalize=True).iloc[0] * 100
                  if len(s) else 0
    )

    return top_pct[top_pct >= threshold]

tvs: pd.DataFrame = top_value_share(df, threshold=98)
df.drop(tvs.index, axis=1, inplace=True)

In [44]:
# one hot encode data
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_cols = df.select_dtypes(include=['category']).columns.tolist()
cat_cols.remove('Instrument')
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
int_cols = df.select_dtypes(include=['int']).columns.tolist()
df[int_cols] = df[int_cols].astype('Float64')
df[bool_cols] = df[bool_cols].astype('bool')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, dtype='bool'), cat_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform='pandas')
df = preprocessor.fit_transform(df)

In [49]:
df.to_csv(config.data_dir / 'datasets' / 'median' / 'median_imputed_encoded.csv')

dtypes: pd.DataFrame = df.dtypes.to_frame('dtypes').reset_index()
dtypes.to_csv(config.data_dir / "datasets" / "median" / "median_imputed_dtypes.csv")